# Data Preparation

Downloads and saves the raw datasets used by all experiment notebooks.  
(Available on `huggingface.co/datasets/PHMGC/roberta-bias-reduction-datasets`)

1. **Ribeiro et al. [A–J]** — 10 naturally imbalanced sentiment datasets (IR 1.41–6.60).
2. **McAuley Lab [K–M]** — 3 naturally imbalanced Amazon review datasets (IR 10.73–39.71).

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.download_utils import download_raw_dataset
from src.data_utils import compute_class_distribution, save_class_distribution

/home/phmgc/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Ribeiro et al. [A–J]

Binary sentiment datasets from SentiBench (Ribeiro et al., 2016).  
Each dataset folder contains `texts.txt` (one text per line) and `score.txt` (labels: `-1` or `1`).  
Binarization: `-1` → `0`, `1` → `1`.

In [3]:
ribeiro_datasets = [
    #"sentistrength_twitter",   # A — IR 1.41
    #"vader_amazon",            # B — IR 1.44
    #"english_dailabor",        # C — IR 1.51
    #"debate",                  # D — IR 1.71
    #"sentistrength_youtube",   # E — IR 2.17
    #"vader_twitter",           # F — IR 2.23
    #"tweet_semevaltest",       # G — IR 2.66
    #"sentistrength_digg",      # H — IR 2.72
    "sentistrength_myspace",   # I — IR 5.32
    #"sentistrength_bbc",       # J — IR 6.60
]

for name in ribeiro_datasets:
    ds = download_raw_dataset("ribeiro", name)
    dist = save_class_distribution("ribeiro", name, ds)
    total = sum(dist.values())
    print(f"  {name:30s}  total={total:,}  " + "  ".join(f"class {k}={v:,} ({v/total:.0%})" for k, v in sorted(dist.items())))

Saving the dataset (1/1 shards): 100%|██████████| 834/834 [00:00<00:00, 27378.56 examples/s]

Salvo: /home/phmgc/tp/Oversampling-LLM-Bias-Reduction/data/raw/ribeiro/sentistrength_myspace
  sentistrength_myspace           total=834  class 0=132 (16%)  class 1=702 (84%)


## 2. McAuley Lab [K–M]

Naturally imbalanced Amazon review datasets.  
Binarization: rating ∈ {1,2} → 0 (negative), rating ∈ {4,5} → 1 (positive), rating 3 discarded.

In [4]:
mcauley_datasets = [
    "luxury_beauty",   # K
    "cds_reviews",     # L
    "digital_music",   # M
]

for name in mcauley_datasets:
    ds = download_raw_dataset("mcauley", name)
    dist = save_class_distribution("mcauley", name, ds)
    total = sum(dist.values())
    print(f"  {name}: total={total:,} | " + " | ".join(f"class {k}={v:,} ({v/total:.0%})" for k, v in sorted(dist.items())))

  Baixando luxury_beauty
  7 MB / 7 MB (100.0%)
  Processando luxury_beauty

  30,385 registros no total


Generating train split: 30385 examples [00:00, 316766.24 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 30385/30385 [00:04<00:00, 7144.01 examples/s]


  Salvo: /home/phmgc/tp/Oversampling-LLM-Bias-Reduction/data/raw/mcauley/luxury_beauty
  luxury_beauty: total=30,385 | class 0=2,591 (9%) | class 1=27,794 (91%)
  Baixando cds_reviews
  484 MB / 484 MB (100.0%)
  Processando cds_reviews
  1,300,000 registros processados
  1,333,070 registros no total


Generating train split: 1333070 examples [00:06, 220368.79 examples/s]
Saving the dataset (2/2 shards): 100%|██████████| 1333070/1333070 [00:10<00:00, 129708.14 examples/s]


  Salvo: /home/phmgc/tp/Oversampling-LLM-Bias-Reduction/data/raw/mcauley/cds_reviews
  cds_reviews: total=1,333,070 | class 0=89,858 (7%) | class 1=1,243,212 (93%)
  Baixando digital_music
  19 MB / 19 MB (100.0%)
  Processando digital_music
  150,000 registros processados
  162,831 registros no total


Generating train split: 162831 examples [00:00, 841858.44 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 162831/162831 [00:00<00:00, 223424.44 examples/s]


  Salvo: /home/phmgc/tp/Oversampling-LLM-Bias-Reduction/data/raw/mcauley/digital_music
  digital_music: total=162,831 | class 0=4,004 (2%) | class 1=158,827 (98%)


## Replication Notes

Decisions made to align this pipeline with the original paper's experimental setup:

### Dataset versions (McAuley Lab)
The paper uses the **5-core** filtered subsets of the Amazon review datasets, not the full corpora. The 5-core filter retains only reviews from users and products with at least 5 interactions, yielding substantially smaller and more representative datasets.

| Dataset | Full size | 5-core size | IR (full) | IR (5-core / paper) |
|---|---|---|---|---|
| luxury_beauty | 532,253 | 30,385 | 5.65 | 10.73 |
| digital_music | 1,525,296 | 162,831 | 21.9 | 39.67 |
| cds_reviews | 1,400,000 | 1,333,070 | 12.3 | 13.84 |